<a href="https://colab.research.google.com/github/BASE-Laboratory/OpenImpala/blob/master/tutorials/05_surrogate_modelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 5: The AI Data Factory -- High-Throughput Surrogate Modeling

**The Pain Point:** Training Machine Learning models to predict material properties requires thousands of labeled 3D structures with ground-truth physics. Generating that data with standard solvers would take months of compute time.

**The Solution:** We use OpenImpala as an automated "Data Factory", generating randomized microstructures and evaluating their physics in seconds. This produces a labeled dataset large enough to train a surrogate model.

**Connection to Tutorial 3 (REV):** In the [REV tutorial](03_rev_and_uncertainty.ipynb), we showed that small sub-volumes have high variance in transport properties, and that variance is physically meaningful. Here, we *exploit* that variance: each small crop with its unique porosity and tortuosity becomes a training sample. The REV variance from Tutorial 3 directly maps to the spread of training data that makes our surrogate model robust.

**In this tutorial we will:**
1. Generate 50 randomized microstructures.
2. Measure their morphological properties (Porosity, Surface Area).
3. Use OpenImpala to calculate their "Ground Truth" Tortuosity.
4. Train a Random Forest model to predict Tortuosity 1000x faster.

In [ ]:
# Install OpenImpala and ML libraries
!pip install -q openimpala porespy scikit-learn matplotlib

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import porespy as ps
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import openimpala as oi

print(f"OpenImpala version {oi.__version__} loaded.")

N_SAMPLES = 50
SIZE = 32

In [ ]:
print("--- Step 1: Generating Data with OpenImpala ---")
X_features = []  # Will hold [Porosity, Specific Surface Area]
y_target = []    # Will hold [Tortuosity]

t_start = time.time()

# Wrap the entire generation loop in one Session to avoid MPI init/finalize overhead
with oi.Session():
    for i in range(N_SAMPLES):
        # 1. Generate random blobby structure (varying porosity and blob size)
        target_porosity = np.random.uniform(0.3, 0.7)
        blobiness = np.random.uniform(1.0, 3.0)
        micro = ps.generators.blobs(shape=[SIZE, SIZE, SIZE], porosity=target_porosity, blobiness=blobiness)
        micro = micro.astype(np.int32)

        # 2. Extract quick morphological features
        vf = oi.volume_fraction(micro, phase=1).fraction
        surface_area = ps.metrics.specific_surface_area(micro)

        # 3. Get Ground Truth physics from OpenImpala
        perc = oi.percolation_check(micro, phase=1, direction="z")
        if perc.percolates:
            tau = oi.tortuosity(micro, phase=1, direction="z").tortuosity
            X_features.append([vf, surface_area])
            y_target.append(tau)

X_features = np.array(X_features)
y_target = np.array(y_target)

t_data = time.time() - t_start
print(f"Successfully generated {len(y_target)} valid samples in {t_data:.2f} seconds.")

In [ ]:
print("\n--- Step 2: Training the AI Surrogate Model ---")
X_train, X_test, y_train, y_test = train_test_split(X_features, y_target, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Make predictions
t0 = time.time()
y_pred = model.predict(X_test)
t_predict = time.time() - t0

r2 = r2_score(y_test, y_pred)
print(f"Model R^2 Score: {r2:.3f}")
print(f"Time to predict {len(X_test)} structures: {t_predict:.4f} seconds")

# Plot True vs Predicted
fig, ax = plt.subplots(figsize=(6, 5), dpi=120)
ax.scatter(y_test, y_pred, color='#1f77b4', s=50, alpha=0.8, edgecolor='black')
ax.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], 'r--', lw=2, label="Perfect Prediction")
ax.set_xlabel("True Tortuosity (OpenImpala)", fontweight='bold')
ax.set_ylabel("Predicted Tortuosity (Random Forest)", fontweight='bold')
ax.set_title("AI Surrogate Model Accuracy", fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## What's Next?

You just used OpenImpala as an automated physics engine to generate labeled training data, then trained an ML model that predicts tortuosity from morphological features alone.

**Continue the journey:**
- [Tutorial 6: Topology Optimisation](06_topology_optimisation.ipynb) -- Use OpenImpala inside a generative design loop to *invent* the optimal microstructure.
- [Tutorial 7: Laptop to Supercomputer](07_hpc_scaling.ipynb) -- Scale up to massive datasets with MPI.